In [9]:
import dotenv
import psycopg2
import pandas as pd
import os
from dotenv import load_dotenv
import glob
load_dotenv()
import random
import numpy as np
from io import StringIO


OUTPUT_DIR = "./export_supabase/vtc_test"
OUTPUT_DIR_BBDD = "./export_supabase/vtc_test_BBDD"

In [5]:
driver_reviews = pd.read_csv(f"{OUTPUT_DIR_BBDD}/driver_reviews.csv")
locations = pd.read_csv(f"{OUTPUT_DIR_BBDD}/locations.csv")
payments = pd.read_csv(f"{OUTPUT_DIR_BBDD}/payments.csv")
promotions = pd.read_csv(f"{OUTPUT_DIR_BBDD}/promotions.csv")
rides = pd.read_csv(f"{OUTPUT_DIR_BBDD}/rides.csv")
users = pd.read_csv(f"{OUTPUT_DIR_BBDD}/users.csv")
vehicles = pd.read_csv(f"{OUTPUT_DIR_BBDD}/vehicles.csv")

print("Lenght driver_reviews:", len(driver_reviews))
print("Lenght locations:", len(locations))
print("Lenght payments:", len(payments))
print("Lenght promotions:", len(promotions))
print("Lenght rides:", len(rides))
print("Lenght users:", len(users))
print("Lenght vehicles:", len(vehicles))

Lenght driver_reviews: 100320
Lenght locations: 20000
Lenght payments: 500000
Lenght promotions: 20
Lenght rides: 500000
Lenght users: 100000
Lenght vehicles: 10000


In [8]:
driver_reviews.head()

,id,driver_id,customer_id,rating,comment,review_date
0,1,2,1,5.0,"Great ride, very friendly driver!",2026-04-20 20:08:45.897727+00:00
1,2,4,3,4.5,"Smooth ride, but the car was a bit messy.",2026-04-20 20:08:45.897727+00:00


In [9]:
locations.head()

,id,address,latitude,longitude
0,1,"123 Main St, Anytown, USA",40.7128,-74.0060
1,2,"456 Elm St, Othertown, USA",34.0522,-118.2437


In [10]:
payments.head()

,id,ride_id,amount,payment_method,payment_time,transaction_id
0,1,1,25.0,credit card,2026-04-20 20:08:45.897727+00:00,TXN123456
1,2,2,30.0,paypal,2026-04-20 20:08:45.897727+00:00,TXN789012


In [11]:
promotions.head()

,id,code,description,discount_percentage,valid_from,valid_to
0,1,WELCOME10,10% off for new users,10.0,2023-09-01,2023-12-31
1,2,SAVE20,20% off on rides over $50,20.0,2023-10-01,2023-11-30


In [12]:
rides.head()

,id,customer_id,driver_id,vehicle_id,pickup_location_id,dropoff_location_id,start_time,end_time,status,distance,fare
0,1,1,2,1,1,2,2023-10-01 08:00:00+00:00,2023-10-01 08:30:00+00:00,completed,15.5,25.0
1,2,3,4,2,2,1,2023-10-02 09:00:00+00:00,2023-10-02 09:45:00+00:00,completed,20.0,30.0


In [13]:
users.head()

,id,name,email,phone_number,user_type,created_at,rating,profile_picture
0,1,Alice Johnson,alice@example.com,123-456-7890,customer,2026-04-20 20:08:45.897727+00:00,4.5,alice.jpg
1,2,Bob Smith,bob@example.com,234-567-8901,driver,2026-04-20 20:08:45.897727+00:00,4.8,bob.jpg
2,3,Charlie Brown,charlie@example.com,345-678-9012,customer,2026-04-20 20:08:45.897727+00:00,4.2,charlie.jpg
3,4,David Wilson,david@example.com,456-789-0123,driver,2026-04-20 20:08:45.897727+00:00,4.9,david.jpg


In [14]:
vehicles.head()

,id,driver_id,make,model,year,license_plate,color,insurance_expiry
0,1,2,Toyota,Corolla,2018,ABC123,Blue,2024-05-01
1,2,4,Honda,Civic,2020,XYZ789,Red,2025-08-15


## Creamos información random

In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

np.random.seed(42)
random.seed(42)
# 1) users
n_users = 100_000
n_drivers = 10_000

user_ids = np.arange(1, n_users + 1)
user_types = np.array(['customer'] * n_users)
user_types[:n_drivers] = 'driver'
np.random.shuffle(user_types)

users = pd.DataFrame({
    "id": user_ids,
    "name": [f"User {i}" for i in user_ids],
    "email": [f"user{i}@example.com" for i in user_ids],
    "phone_number": [f"{1000000000 + i}" for i in user_ids],
    "user_type": user_types,
    "created_at": pd.to_datetime("2023-01-01") + pd.to_timedelta(np.random.randint(0, 365, n_users), unit="D"),
    "rating": np.round(3.5 + np.random.rand(n_users) * 1.5, 1),
    "profile_picture": [f"user{i}.jpg" for i in user_ids],
})
users.to_csv(f"{OUTPUT_DIR}/users.csv", index=False)

# 2) vehicles (uno por driver)
drivers = users[users["user_type"] == "driver"].copy()
drivers = drivers.sort_values("id").reset_index(drop=True)
n_vehicles = len(drivers)

makes = ['Toyota','Honda','Ford','Hyundai','Kia','Volkswagen','Seat','Renault']
models = ['Corolla','Civic','Focus','i30','Ceed','Golf','Ibiza','Clio']
colors = ['Blue','Red','Black','White','Silver','Grey']

vehicles = pd.DataFrame({
    "id": np.arange(1, n_vehicles + 1),
    "driver_id": drivers["id"].values,
    "make": np.random.choice(makes, n_vehicles),
    "model": np.random.choice(models, n_vehicles),
    "year": np.random.randint(2015, 2023, n_vehicles),
    "license_plate": [f"PLT{1000 + i:04d}" for i in range(n_vehicles)],
    "color": np.random.choice(colors, n_vehicles),
    "insurance_expiry": pd.to_datetime("2024-01-01") + pd.to_timedelta(np.random.randint(0, 365, n_vehicles), unit="D"),
})
vehicles.to_csv(f"{OUTPUT_DIR}/vehicles.csv", index=False)

# 3) locations
n_locations = 20_000
locations = pd.DataFrame({
    "id": np.arange(1, n_locations + 1),
    "address": [f"Calle {i}, Ciudad {i % 50 + 1}" for i in range(1, n_locations + 1)],
    "latitude": 40.0 + np.random.rand(n_locations) * 0.5,
    "longitude": -3.8 + np.random.rand(n_locations) * 0.5,
})
locations.to_csv(f"{OUTPUT_DIR}/locations.csv", index=False)

# 4) promotions
n_promos = 20
discounts = [5.0, 10.0, 15.0, 20.0]
promotions = pd.DataFrame({
    "id": np.arange(1, n_promos + 1),
    "code": [f"PROMO{i:03d}" for i in range(1, n_promos + 1)],
    "description": [f"Promo auto {i}" for i in range(1, n_promos + 1)],
    "discount_percentage": np.random.choice(discounts, n_promos),
    "valid_from": pd.to_datetime("2023-01-01") + pd.to_timedelta(np.random.randint(0, 180, n_promos), unit="D"),
    "valid_to": pd.to_datetime("2023-07-01") + pd.to_timedelta(np.random.randint(0, 180, n_promos), unit="D"),
})
promotions.to_csv(f"{OUTPUT_DIR}/promotions.csv", index=False)

# 5) rides
n_rides = 500_000
customers = users[users["user_type"] == "customer"].copy()

def sample_times(n):
    base_dates = pd.to_datetime("2023-09-01") + pd.to_timedelta(np.random.randint(0, 90, n), unit="D")
    peak_hours = np.array([7,8,9,17,18,19,20])
    off_hours = np.array([10,11,12,13,14,15,16,21,22,23])
    hours = []
    for _ in range(n):
        if random.random() < 0.6:
            hours.append(int(np.random.choice(peak_hours)))
        else:
            hours.append(int(np.random.choice(off_hours)))
    minutes = np.random.randint(0, 60, n)
    start = base_dates + pd.to_timedelta(hours, unit="h") + pd.to_timedelta(minutes, unit="m")
    durations = np.random.randint(10, 50, n)  # minutos
    end = start + pd.to_timedelta(durations, unit="m")
    return start, end

ride_start, ride_end = sample_times(n_rides)

pickup_ids = np.random.randint(1, n_locations + 1, n_rides)
dropoff_ids = np.random.randint(1, n_locations + 1, n_rides)
same = pickup_ids == dropoff_ids
dropoff_ids[same] = (dropoff_ids[same] % n_locations) + 1

driver_ids = np.random.choice(drivers["id"].values, n_rides)
vehicle_map = vehicles.set_index("driver_id")["id"].to_dict()
vehicle_ids = [vehicle_map[d] for d in driver_ids]

distance = np.round(2.0 + np.random.rand(n_rides) * 23.0, 1)
fare = np.round(2.0 + distance * (0.7 + np.random.rand(n_rides) * 0.6) + np.random.rand(n_rides) * 3.0, 2)

# Lista de códigos PROMO
promo_codes = [f"PROMO{i:03d}" for i in range(1, 21)]

# Lista de códigs de tarifa

fares = [f"c_{i}" for i in range(1,50)]

# Probabilidad de que un ride tenga promoción (ajústala si quieres)
promo_probability = 0.10  # 10% de los viajes tienen promo

# Generar fare_id aleatorio o None
fare_ids = np.where(
    np.random.rand(n_rides) < promo_probability,
    np.random.choice(promo_codes, n_rides),
    np.random.choice(fares, n_rides)
)

rides = pd.DataFrame({
    "id": np.arange(1, n_rides + 1),
    "customer_id": np.random.choice(customers["id"].values, n_rides),
    "driver_id": driver_ids,
    "vehicle_id": vehicle_ids,
    "pickup_location_id": pickup_ids,
    "dropoff_location_id": dropoff_ids,
    "start_time": ride_start,
    "end_time": ride_end,
    "status": "completed",
    "distance": distance,
    "fare": fare,
    "fare_id": fare_ids
})
rides.to_csv(f"{OUTPUT_DIR}/rides.csv", index=False)

# 6) payments
methods = ["credit card", "paypal", "cash"]
payments = pd.DataFrame({
    "id": np.arange(1, n_rides + 1),
    "ride_id": rides["id"],
    "amount": rides["fare"],
    "payment_method": np.random.choice(methods, n_rides, p=[0.6, 0.25, 0.15]),
    "payment_time": rides["end_time"] + pd.to_timedelta(np.random.randint(0, 10, n_rides), unit="m"),
    "transaction_id": [f"TXN{i:08d}" for i in rides["id"]],
})
payments.to_csv(f"{OUTPUT_DIR}/payments.csv", index=False)

# 7) driver_reviews (~20% de rides)
mask = np.random.rand(n_rides) < 0.2
review_rides = rides[mask].copy()
n_reviews = len(review_rides)

comments = [
    "Great ride, very friendly driver!",
    "Smooth ride, car was clean.",
    "Driver arrived on time.",
    "Good service, would ride again.",
    "Ride was okay, a bit of traffic.",
    "Excellent experience, highly recommended.",
    "Driver was polite and professional."
]

driver_reviews = pd.DataFrame({
    "id": np.arange(1, n_reviews + 1),
    "driver_id": review_rides["driver_id"].values,
    "customer_id": review_rides["customer_id"].values,
    "rating": np.round(3.5 + np.random.rand(n_reviews) * 1.5, 1),
    "comment": np.random.choice(comments, n_reviews),
    "review_date": review_rides["end_time"].values + np.random.randint(0, 120, n_reviews).astype("timedelta64[m]"),
})
driver_reviews.to_csv(f"{OUTPUT_DIR}/driver_reviews.csv", index=False)

# Crear CSV maestro de tarifas

# Lista de fare_id que me pasaste
fare_ids = [
    'c_18', 'c_49', 'c_19', 'c_23', 'c_47', 'c_14', 'c_6', 'c_13',
    'c_20', 'c_3', 'c_36', 'c_4', 'c_34', 'c_28', 'c_1', 'c_26',
    'c_29', 'c_31', 'PROMO018', 'c_45', 'PROMO020', 'c_24', 'c_37',
    'c_8', 'c_21', 'PROMO012', 'c_30', 'PROMO015', 'c_9', 'c_39',
    'c_7', 'c_27', 'c_17', 'PROMO011', 'PROMO016', 'PROMO009', 'c_48',
    'c_32', 'c_42', 'PROMO013', 'c_5', 'c_15', 'c_33', 'c_22',
    'PROMO017', 'c_44', 'PROMO010', 'c_43', 'c_46', 'c_35', 'c_38',
    'c_25', 'c_2', 'c_40', 'PROMO005', 'PROMO006', 'c_12', 'PROMO002',
    'c_11', 'PROMO008', 'c_41', 'c_10', 'PROMO007', 'PROMO019',
    'PROMO003', 'PROMO004', 'c_16', 'PROMO001', 'PROMO014'
]

# Lista de nombres de tarifas realistas
fare_names_pool = [
    "Tarifa Estándar", "Tarifa Estándar Plus", "Tarifa Básica", "Tarifa Básica Plus",
    "Tarifa Económica", "Tarifa Económica Plus", "Tarifa Media Distancia",
    "Tarifa Media Distancia Plus", "Tarifa Larga Distancia", "Tarifa Larga Distancia Plus",
    "Tarifa Nocturna", "Tarifa Nocturna Plus", "Tarifa Día Laboral", "Tarifa Día Laboral Plus",
    "Tarifa Festivos", "Tarifa Festivos Plus", "Tarifa Aeropuerto", "Tarifa Aeropuerto Plus",
    "Tarifa Aeropuerto XL", "Tarifa XL", "Tarifa XL Plus", "Tarifa Flexible",
    "Tarifa Flexible Plus", "Tarifa Flexible XL", "Tarifa Premium", "Tarifa Premium Plus",
    "Tarifa Business", "Tarifa Business Plus", "Tarifa Corporativa", "Tarifa Corporativa Plus",
    "Tarifa Express", "Tarifa Rápida", "Tarifa Rápida Plus", "Tarifa Confort",
    "Tarifa Especial", "Tarifa Especial Plus", "Tarifa Dinámica", "Tarifa Dinámica Plus",
    "Tarifa Dinámica XL", "Tarifa Alta Demanda", "Tarifa Urbana", "Tarifa Urbana Plus"
]

# Si hay más fare_id que nombres, repetimos aleatoriamente
fare_names = np.random.choice(fare_names_pool, size=len(fare_ids), replace=True)

# Crear DataFrame maestro
fare_master = pd.DataFrame({
    "fare_id": fare_ids,
    "fare_name": fare_names
})

# Guardar CSV
fare_master.to_csv("fare_master.csv", index=False)

print("CSV generado: fare_master.csv")
print(fare_master.head())



CSV generado: fare_master.csv
  fare_id                    fare_name
0    c_18  Tarifa Media Distancia Plus
1    c_49            Tarifa Aeropuerto
2    c_19          Tarifa Premium Plus
3    c_23              Tarifa Flexible
4    c_47           Tarifa Urbana Plus


## Crear más datos para rides

In [4]:
# ---------------------------------------------------------
# 1. Función para generar fechas realistas en un rango
# ---------------------------------------------------------
def sample_times(n, start_date, end_date):
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    total_days = (end_date - start_date).days

    # Día aleatorio dentro del rango
    base_dates = start_date + pd.to_timedelta(
        np.random.randint(0, total_days, n), unit="D"
    )

    peak_hours = np.array([7,8,9,17,18,19,20])
    off_hours = np.array([10,11,12,13,14,15,16,21,22,23])

    hours = np.where(
        np.random.rand(n) < 0.6,
        np.random.choice(peak_hours, n),
        np.random.choice(off_hours, n)
    )

    minutes = np.random.randint(0, 60, n)

    start = base_dates + pd.to_timedelta(hours, unit="h") + pd.to_timedelta(minutes, unit="m")
    durations = np.random.randint(10, 50, n)
    end = start + pd.to_timedelta(durations, unit="m")

    return start, end


# ---------------------------------------------------------
# 2. Parámetros
# ---------------------------------------------------------
n_rides_2024 = 500_000
n_rides_2025 = 500_000

# Lista de fare_id (de tu maestro)
fare_ids = fare_master["fare_id"].values


# ---------------------------------------------------------
# 3. Generar fechas para 2024 y 2025
# ---------------------------------------------------------
ride_start_2024, ride_end_2024 = sample_times(n_rides_2024, "2024-01-01", "2024-12-31")
ride_start_2025, ride_end_2025 = sample_times(n_rides_2025, "2025-01-01", "2025-12-31")


# ---------------------------------------------------------
# 4. Evitar colisiones de ID
# ---------------------------------------------------------
# Obtener el último ID existente en rides
last_existing_id = rides["id"].max()

start_id_2024 = last_existing_id + 1
start_id_2025 = start_id_2024 + n_rides_2024


# ---------------------------------------------------------
# 5. Generar rides 2024
# ---------------------------------------------------------
driver_ids_2024 = np.random.choice(drivers["id"].values, n_rides_2024)
vehicle_map = vehicles.set_index("driver_id")["id"].to_dict()
vehicle_ids_2024 = [vehicle_map[d] for d in driver_ids_2024]

pickup_ids_2024 = np.random.randint(1, n_locations + 1, n_rides_2024)
dropoff_ids_2024 = np.random.randint(1, n_locations + 1, n_rides_2024)
same = pickup_ids_2024 == dropoff_ids_2024
dropoff_ids_2024[same] = (dropoff_ids_2024[same] % n_locations) + 1

distance_2024 = np.round(2.0 + np.random.rand(n_rides_2024) * 23.0, 1)
fare_2024 = np.round(2.0 + distance_2024 * (0.7 + np.random.rand(n_rides_2024) * 0.6) + np.random.rand(n_rides_2024) * 3.0, 2)

rides_2024 = pd.DataFrame({
    "id": np.arange(start_id_2024, start_id_2024 + n_rides_2024),
    "customer_id": np.random.choice(customers["id"].values, n_rides_2024),
    "driver_id": driver_ids_2024,
    "vehicle_id": vehicle_ids_2024,
    "pickup_location_id": pickup_ids_2024,
    "dropoff_location_id": dropoff_ids_2024,
    "start_time": ride_start_2024,
    "end_time": ride_end_2024,
    "status": "completed",
    "distance": distance_2024,
    "fare": fare_2024,
    "fare_id": np.random.choice(fare_ids, n_rides_2024)
})


# ---------------------------------------------------------
# 6. Generar rides 2025
# ---------------------------------------------------------
driver_ids_2025 = np.random.choice(drivers["id"].values, n_rides_2025)
vehicle_ids_2025 = [vehicle_map[d] for d in driver_ids_2025]

pickup_ids_2025 = np.random.randint(1, n_locations + 1, n_rides_2025)
dropoff_ids_2025 = np.random.randint(1, n_locations + 1, n_rides_2025)
same = pickup_ids_2025 == dropoff_ids_2025
dropoff_ids_2025[same] = (dropoff_ids_2025[same] % n_locations) + 1

distance_2025 = np.round(2.0 + np.random.rand(n_rides_2025) * 23.0, 1)
fare_2025 = np.round(2.0 + distance_2025 * (0.7 + np.random.rand(n_rides_2025) * 0.6) + np.random.rand(n_rides_2025) * 3.0, 2)

rides_2025 = pd.DataFrame({
    "id": np.arange(start_id_2025, start_id_2025 + n_rides_2025),
    "customer_id": np.random.choice(customers["id"].values, n_rides_2025),
    "driver_id": driver_ids_2025,
    "vehicle_id": vehicle_ids_2025,
    "pickup_location_id": pickup_ids_2025,
    "dropoff_location_id": dropoff_ids_2025,
    "start_time": ride_start_2025,
    "end_time": ride_end_2025,
    "status": "completed",
    "distance": distance_2025,
    "fare": fare_2025,
    "fare_id": np.random.choice(fare_ids, n_rides_2025)
})


# ---------------------------------------------------------
# 7. Exportar a CSV
# ---------------------------------------------------------
# rides_2024.to_csv(f"{OUTPUT_DIR}/rides_2024.csv", index=False)
# rides_2025.to_csv(f"{OUTPUT_DIR}/rides_2025.csv", index=False)

# print("CSV generados: rides_2024.csv y rides_2025.csv")


In [16]:
driver_reviews = pd.read_csv(f"{OUTPUT_DIR}/driver_reviews.csv")
locations = pd.read_csv(f"{OUTPUT_DIR}/locations.csv")
payments = pd.read_csv(f"{OUTPUT_DIR}/payments.csv")
promotions = pd.read_csv(f"{OUTPUT_DIR}/promotions.csv")
rides = pd.read_csv(f"{OUTPUT_DIR}/rides.csv")
users = pd.read_csv(f"{OUTPUT_DIR}/users.csv")
vehicles = pd.read_csv(f"{OUTPUT_DIR}/vehicles.csv")
fare_master = pd.read_csv(f"{OUTPUT_DIR}/fare_master.csv")

print("Lenght driver_reviews:", len(driver_reviews))
print("Lenght locations:", len(locations))
print("Lenght payments:", len(payments))
print("Lenght promotions:", len(promotions))
print("Lenght rides:", len(rides))
print("Lenght users:", len(users))
print("Lenght vehicles:", len(vehicles))
print("Lenght fare master:", len(fare_master))

Lenght driver_reviews: 100278
Lenght locations: 20000
Lenght payments: 500000
Lenght promotions: 20
Lenght rides: 500000
Lenght users: 100000
Lenght vehicles: 10000
Lenght fare master: 69


In [ ]:
fare_master[fare_master['fare_n']]==fare_master['fare_ids'].unique()

array(['Tarifa Media Distancia Plus', 'Tarifa Aeropuerto',
       'Tarifa Premium Plus', 'Tarifa Flexible', 'Tarifa Urbana Plus',
       'Tarifa Aeropuerto Plus', 'Tarifa Flexible Plus',
       'Tarifa Especial', 'Tarifa Corporativa', 'Tarifa XL Plus',
       'Tarifa Básica', 'Tarifa Business Plus', 'Tarifa Confort',
       'Tarifa Económica Plus', 'Tarifa Dinámica',
       'Tarifa Larga Distancia', 'Tarifa Corporativa Plus',
       'Tarifa Alta Demanda', 'Tarifa Larga Distancia Plus',
       'Tarifa Especial Plus', 'Tarifa Dinámica XL', 'Tarifa Flexible XL',
       'Tarifa Día Laboral Plus', 'Tarifa Media Distancia',
       'Tarifa Express', 'Tarifa Urbana', 'Tarifa Básica Plus',
       'Tarifa Nocturna', 'Tarifa Estándar Plus', 'Tarifa Estándar',
       'Tarifa XL', 'Tarifa Festivos'], dtype=object)

In [13]:
rides['fare_id'].unique()

array(['c_18', 'c_49', 'c_19', 'c_23', 'c_47', 'c_14', 'c_6', 'c_13',
       'c_20', 'c_3', 'c_36', 'c_4', 'c_34', 'c_28', 'c_1', 'c_26',
       'c_29', 'c_31', 'PROMO018', 'c_45', 'PROMO020', 'c_24', 'c_37',
       'c_8', 'c_21', 'PROMO012', 'c_30', 'PROMO015', 'c_9', 'c_39',
       'c_7', 'c_27', 'c_17', 'PROMO011', 'PROMO016', 'PROMO009', 'c_48',
       'c_32', 'c_42', 'PROMO013', 'c_5', 'c_15', 'c_33', 'c_22',
       'PROMO017', 'c_44', 'PROMO010', 'c_43', 'c_46', 'c_35', 'c_38',
       'c_25', 'c_2', 'c_40', 'PROMO005', 'PROMO006', 'c_12', 'PROMO002',
       'c_11', 'PROMO008', 'c_41', 'c_10', 'PROMO007', 'PROMO019',
       'PROMO003', 'PROMO004', 'c_16', 'PROMO001', 'PROMO014'],
      dtype=object)

In [14]:
rides.head()

,id,customer_id,driver_id,vehicle_id,pickup_location_id,dropoff_location_id,start_time,end_time,status,distance,fare,fare_id
0,1,29059,97708,9751,589,4677,2023-10-16 21:50:00,2023-10-16 22:29:00,completed,7.8,10.84,c_18
1,2,93797,2933,300,2315,1536,2023-09-08 09:21:00,2023-09-08 09:57:00,completed,23.5,21.31,c_49
2,3,74763,88005,8773,11082,8494,2023-11-16 09:52:00,2023-11-16 10:34:00,completed,3.6,7.10,c_19
3,4,31131,42193,4196,8474,17004,2023-11-11 18:29:00,2023-11-11 18:52:00,completed,13.2,19.56,c_23
4,5,86453,21169,2107,1476,3406,2023-10-18 10:55:00,2023-10-18 11:19:00,completed,20.9,28.82,c_47


In [ ]:
host = os.getenv("HOST"),
port = os.getenv("PORT"),
database = os.getenv("DB"),
user = os.getenv("USER"),
password = os.getenv("PASSWORD")

In [34]:
len(driver_reviews)

100320

In [35]:
driver_reviews.head()

,id,driver_id,customer_id,rating,comment,review_date
0,1,26424,66703,3.6,Driver arrived on time.,2023-11-26 22:55:00
1,2,71992,54223,4.8,"Ride was okay, a bit of traffic.",2023-11-12 17:31:00
2,3,71767,90599,4.2,"Smooth ride, car was clean.",2023-10-10 14:38:00
3,4,67711,8385,4.2,Driver arrived on time.,2023-09-24 07:45:00
4,5,80349,45760,3.8,Driver arrived on time.,2023-10-30 15:46:00


In [36]:
len(driver_reviews['driver_id'].unique())

9999

In [17]:
## Subir las tablas a supabase

conn = psycopg2.connect(
    host= os.getenv("HOST"),
    port= os.getenv("PORT"),
    database= os.getenv("DB"),
    user= os.getenv("USER"),
    password= os.getenv("PASSWORD"),
    connect_timeout = 0, ## Desactivamos el timeout para conexiones largas
    options = '-c statement_timeout=0' ## Desactivamos el timeout para consultas largas
)

cur = conn.cursor()

with open(f"{OUTPUT_DIR}/fare_master.csv", "r", encoding="utf-8") as f:
    cur.copy_expert("COPY public.fare_master FROM STDIN WITH CSV HEADER", f)

conn.commit()
cur.close()
conn.close()


## Subir archivos grandes creando particiones en CSV

In [19]:
##Subir archivos grandes dividiendolos en partes
INPUT = f"{OUTPUT_DIR}/rides.csv"
CHUNK_SIZE = 50000

#Troceamos el archivo en partes de 50k filas y las guardamos en el mismo directorio
i = 0
for chunk in pd.read_csv(INPUT, chunksize=CHUNK_SIZE):
    chunk.to_csv(f"{OUTPUT_DIR}/rides_part_{i}.csv", index=False)
    print(f"Generado rides_part_{i}.csv")
    i += 1


def load_chunk(table, filename):
    conn = psycopg2.connect(
        host=os.getenv("HOST"),
        port=os.getenv("PORT"),
        database=os.getenv("DB"),
        user=os.getenv("USER"),
        password=os.getenv("PASSWORD"),
        connect_timeout=0,
        options='-c statement_timeout=0'
    )
    cur = conn.cursor()
    print(f"Subiendo {filename} ...")

    with open(filename, "r", encoding="utf-8") as f:
        cur.copy_expert(f"COPY public.{table} FROM STDIN WITH CSV HEADER", f)

    conn.commit()
    cur.close()
    conn.close()
    print(f"✔ {filename} subido correctamente")

# Detectar automáticamente todos los chunks
chunk_files = sorted(glob.glob(f"{OUTPUT_DIR}/rides_part_*.csv"))
print(f"Encontrados {len(chunk_files)} archivos de rides")  

# Subir todas las partes
for i in range(10):  # ajusta según cuántos archivos tengas
    load_chunk("rides", f"{OUTPUT_DIR}/rides_part_{i}.csv")


Generado rides_part_0.csv
Generado rides_part_1.csv
Generado rides_part_2.csv
Generado rides_part_3.csv
Generado rides_part_4.csv
Generado rides_part_5.csv
Generado rides_part_6.csv
Generado rides_part_7.csv
Generado rides_part_8.csv
Generado rides_part_9.csv
Encontrados 10 archivos de rides
Subiendo ./export_supabase/vtc_test/rides_part_0.csv ...
✔ ./export_supabase/vtc_test/rides_part_0.csv subido correctamente
Subiendo ./export_supabase/vtc_test/rides_part_1.csv ...
✔ ./export_supabase/vtc_test/rides_part_1.csv subido correctamente
Subiendo ./export_supabase/vtc_test/rides_part_2.csv ...
✔ ./export_supabase/vtc_test/rides_part_2.csv subido correctamente
Subiendo ./export_supabase/vtc_test/rides_part_3.csv ...
✔ ./export_supabase/vtc_test/rides_part_3.csv subido correctamente
Subiendo ./export_supabase/vtc_test/rides_part_4.csv ...
✔ ./export_supabase/vtc_test/rides_part_4.csv subido correctamente
Subiendo ./export_supabase/vtc_test/rides_part_5.csv ...
✔ ./export_supabase/vtc_test/r

## Subir Dataframes grandes sin crear las particiones en CSV

In [10]:
# ---------------------------------------------------------
# Función para subir un chunk de DataFrame a Supabase
# ---------------------------------------------------------
def upload_chunk(df_chunk, table_name):
    # Convertir chunk a CSV en memoria
    buffer = StringIO()
    df_chunk.to_csv(buffer, index=False)
    buffer.seek(0)

    # Conexión al pooler (sin timeout)
    conn = psycopg2.connect(
        host=os.getenv("HOST"),
        port=os.getenv("PORT"),
        database=os.getenv("DB"),
        user=os.getenv("USER"),
        password=os.getenv("PASSWORD"),
        connect_timeout=0,
        options='-c statement_timeout=0'
    )
    cur = conn.cursor()

    print(f"Subiendo chunk de {len(df_chunk)} filas...")

    cur.copy_expert(
        f"COPY public.{table_name} FROM STDIN WITH CSV HEADER",
        buffer
    )

    conn.commit()
    cur.close()
    conn.close()

    print("✔ Chunk subido correctamente")


# ---------------------------------------------------------
# Función para subir un DataFrame grande en chunks
# ---------------------------------------------------------
def upload_dataframe_in_chunks(df, table_name, chunk_size=50_000):
    total = len(df)
    print(f"Subiendo {total:,} filas a {table_name} en chunks de {chunk_size:,}...")

    for start in range(0, total, chunk_size):
        end = start + chunk_size
        df_chunk = df.iloc[start:end]
        upload_chunk(df_chunk, table_name)

    print("✔ DataFrame completo subido a Supabase")


# ---------------------------------------------------------
# EJEMPLO: subir rides_2024 y rides_2025
# ---------------------------------------------------------

upload_dataframe_in_chunks(rides_2024, "rides", chunk_size=50_000)
upload_dataframe_in_chunks(rides_2025, "rides", chunk_size=50_000)


Subiendo 500,000 filas a rides en chunks de 50,000...
Subiendo chunk de 50000 filas...
✔ Chunk subido correctamente
Subiendo chunk de 50000 filas...
✔ Chunk subido correctamente
Subiendo chunk de 50000 filas...
✔ Chunk subido correctamente
Subiendo chunk de 50000 filas...
✔ Chunk subido correctamente
Subiendo chunk de 50000 filas...
✔ Chunk subido correctamente
Subiendo chunk de 50000 filas...
✔ Chunk subido correctamente
Subiendo chunk de 50000 filas...
✔ Chunk subido correctamente
Subiendo chunk de 50000 filas...
✔ Chunk subido correctamente
Subiendo chunk de 50000 filas...
✔ Chunk subido correctamente
Subiendo chunk de 50000 filas...
✔ Chunk subido correctamente
✔ DataFrame completo subido a Supabase
Subiendo 500,000 filas a rides en chunks de 50,000...
Subiendo chunk de 50000 filas...
✔ Chunk subido correctamente
Subiendo chunk de 50000 filas...
✔ Chunk subido correctamente
Subiendo chunk de 50000 filas...
✔ Chunk subido correctamente
Subiendo chunk de 50000 filas...
✔ Chunk subido